# Notebook 1: Business Understanding & Data Preparation

---

## Business Question

**What's happening with our customers? Are we losing people we should be keeping?**

I've been given a dataset of 5,000 Netflix customers and asked to figure out the churn situation. This is the kind of problem I'd actually encounter working as a data analyst - someone hands you data and says "figure out what's going on."

## Why This Matters

Subscription businesses live or die on retention. Acquiring a new customer costs 5-25x more than keeping an existing one. If churn is high, we're bleeding revenue. If we can understand *why* people leave, we can actually do something about it.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# I like seaborn-v0_8-whitegrid - clean but not boring
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

DATA_PATH = Path('../data/netflix_customer_churn.csv')
df = pd.read_csv(DATA_PATH)

---

## First Look: What Are We Working With?

Before doing anything fancy, I want to understand what this dataset actually contains. How many customers? What information do we have? Is anything obviously broken?

---

In [ ]:
print(f"Total customers: {len(df):,}")
print(f"Features per customer: {df.shape[1]}")
print(f"\nMemory footprint: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("\nFirst few rows:")

In [ ]:
df.head(10)

In [ ]:
# Column breakdown - what do we actually have?
print("Column types:")
print(df.dtypes.to_frame('type'))

**Initial observation:** We have customer demographics (age, gender), subscription details (type, monthly fee), behavioral data (watch hours, login recency), and the target variable (churned). This is a solid foundation for analysis.

---

## Data Quality Check

Real datasets always have issues. Missing values, duplicates, weird outliers. I've learned the hard way to check for these things before trusting any analysis.

---

In [ ]:
# Missing values - the silent killer of analyses
missing = df.isnull().sum()
if missing.sum() > 0:
    print("Missing values found:")
    print(missing[missing > 0])
else:
    print("✓ No missing values - this is actually pretty clean for real-world data")

In [ ]:
# Duplicate check
dupes = df.duplicated().sum()
dup_ids = df['customer_id'].duplicated().sum()

print(f"Duplicate rows: {dupes}")
print(f"Duplicate customer IDs: {dup_ids}")

if dupes == 0 and dup_ids == 0:
    print("✓ No duplicates - each customer appears once")

In [ ]:
# Statistical overview
df.describe().round(2)

**What jumps out:**

- Age ranges 18-70 - reasonable spread for a streaming service
- Watch hours have huge variance (some people watch 0, others 50+) - this could be important
- `last_login_days` goes up to 60+ - meaning some customers haven't opened the app in 2 months
- Monthly fee has 3 distinct values - this matches subscription tiers

Nothing obviously broken. The data seems legitimate.

---

## The Main Event: Churn Rate

This is what we're here to understand. How bad is the situation?

---

In [ ]:
churned_count = df['churned'].sum()
retained_count = len(df) - churned_count
churn_rate = df['churned'].mean()

print(f"Churned customers: {churned_count:,} ({churn_rate*100:.1f}%)")
print(f"Retained customers: {retained_count:,} ({(1-churn_rate)*100:.1f}%)")
print(f"\nThat's roughly 1 in 4 customers leaving.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#2ecc71', '#e74c3c']
df['churned'].value_counts().sort_index().plot(kind='bar', color=colors, ax=ax)

ax.set_xticklabels(['Retained', 'Churned'], rotation=0, fontsize=12)
ax.set_title('Customer Retention Overview', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Customers')

# Add count labels
for i, v in enumerate([retained_count, churned_count]):
    ax.text(i, v + 80, f'{v:,}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

**Key Finding:** ~25% churn rate. This is significant - that's a quarter of our revenue walking out the door. In subscription terms, this is actually pretty high (Netflix's actual churn is around 2-3% monthly, for comparison). But for analysis purposes, a higher churn rate actually helps - more signal to learn from.

---

## Understanding Our Customer Base

Before diving into churn specifically, I want to understand the composition of our customer base. Who are these people?

---

In [ ]:
# Categorical variables breakdown
cat_cols = ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].barh(counts.index, counts.values, color='#3498db', alpha=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12)
    axes[i].set_xlabel('Count')

plt.tight_layout()
plt.show()

**Interesting observations:**

1. **Gender** - fairly balanced, with 'Other' being significant. This could reflect privacy-conscious users or non-binary representation. Worth noting for any gender-specific analysis.

2. **Subscription types** - evenly split across Basic/Standard/Premium. This is actually unusual - typically you'd see more Basic users. Might be a synthetic dataset artifact, or it could indicate a mature market.

3. **Regions** - global distribution across 6 continents. Good for regional analysis.

4. **Devices** - Mobile and TV are most popular. The device someone uses might affect their experience and therefore churn.

5. **Payment methods** - including Crypto is interesting. Modern payment options might correlate with different customer behaviors.

---

## Outlier Detection

Are there extreme values that might distort our analysis? I'm particularly curious about the behavioral metrics.

---

In [ ]:
numeric_cols = ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=30, color='#9b59b6', edgecolor='white', alpha=0.7)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Check for outliers using IQR method
print("Outlier analysis (IQR method):")
print("-" * 50)

for col in ['watch_hours', 'last_login_days', 'avg_watch_time_per_day']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = outliers / len(df) * 100
    print(f"{col}: {outliers} outliers ({pct:.1f}%)")

**Decision:** The distributions look reasonable. Some power users with 50+ hours, some inactive users with 60+ days since login. These aren't errors - they're real customer behaviors. Power users and dormant users might be important segments, so I'm keeping these outliers.

**Limitation:** In a real scenario, I'd want to verify these extreme values with the data engineering team. Are 60+ day inactive users actually still paying? That seems like a churn opportunity.

---

## Feature Engineering

Based on what I've seen, I'm creating some derived features that might help with both analysis and downstream modeling.

---

In [ ]:
# Age groups - easier to analyze than continuous age
df['age_group'] = pd.cut(df['age'], 
                         bins=[17, 25, 35, 45, 55, 70], 
                         labels=['18-25', '26-35', '36-45', '46-55', '56+'])

# Engagement level based on watch hours
# I chose these thresholds based on typical streaming behavior
df['engagement_level'] = pd.cut(df['watch_hours'], 
                                bins=[-1, 5, 15, 30, 100],
                                labels=['Low', 'Medium', 'High', 'Power'])

# Login recency - this might be crucial
df['login_recency'] = pd.cut(df['last_login_days'],
                             bins=[-1, 7, 14, 30, 60, 100],
                             labels=['Active', 'Recent', 'Moderate', 'Inactive', 'Dormant'])

# Customer Lifetime Value estimate (simplified)
# In reality, CLV is more complex, but this gives us a rough value per customer
df['clv_estimate'] = df['monthly_fee'] * 12  # Simplified annual value

# High risk flag based on engagement
df['high_risk_flag'] = ((df['last_login_days'] > 30) & (df['watch_hours'] < 10)).astype(int)

print("New features created:")
print(df[['age_group', 'engagement_level', 'login_recency', 'clv_estimate', 'high_risk_flag']].head(10))

**Why these features?**

- **age_group**: Different age cohorts might have different churn patterns
- **engagement_level**: Categorizing watch behavior for easier analysis
- **login_recency**: My hypothesis is this will be very important for churn
- **clv_estimate**: Puts a dollar value on each customer for business impact analysis
- **high_risk_flag**: Quick filter for customers showing warning signs

---

## Early Correlation Analysis

Before deep-diving into specific variables, let me see what correlates with churn. This will guide my investigation in subsequent notebooks.

---

In [ ]:
# Numeric correlations with churn
numeric_df = df.select_dtypes(include=[np.number])
corr_with_churn = numeric_df.corr()['churned'].drop('churned').sort_values(key=abs, ascending=False)

print("Correlation with churn (sorted by magnitude):")
print(corr_with_churn.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Correlation heatmap
corr_cols = ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 
             'number_of_profiles', 'avg_watch_time_per_day', 'churned']
corr_matrix = df[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

**Initial Hypothesis:**

`last_login_days` shows the strongest correlation with churn. This makes intuitive sense - customers who haven't logged in recently have already mentally churned, they just haven't cancelled yet.

**Surprising finding:** `watch_hours` and `avg_watch_time_per_day` don't correlate strongly with churn at the aggregate level. I would have expected engagement to be more predictive. This bears investigation - maybe the relationship isn't linear, or there are interaction effects.

---

## Save Prepared Data

The dataset is now enriched with engineered features and ready for deeper analysis.

---

In [ ]:
df.to_csv('../data/prepared_data.csv', index=False)

print(f"Prepared data saved: {df.shape}")
print(f"\nFeatures ready for analysis:")
for col in df.columns:
    print(f"  - {col}")

---

## Summary & Next Steps

### What I Learned

1. **Data quality is solid** - no missing values, no duplicates, clean structure
2. **Churn rate is ~25%** - that's 1 in 4 customers leaving, significant business problem
3. **Login recency appears to be the strongest signal** - customers who haven't logged in recently are more likely to have churned
4. **Customer base is diverse** - spread across regions, devices, subscription tiers, payment methods

### Unexpected Finding

Watch hours don't correlate as strongly with churn as I expected. Some high-engagement users still churn. This suggests engagement alone isn't retention - there might be other factors like content satisfaction or value perception.

### Limitations

- This is a snapshot dataset, not time-series. I can't see how behavior changes over time.
- I don't have data on *why* customers churned (surveys, exit interviews, etc.)
- The even distribution of subscription types might indicate synthetic data

### Next Question to Explore

**In Notebook 2:** I'll investigate the relationship between specific behaviors and churn. Why do some engaged users still leave? What's different about Basic vs Premium churners? Is there a critical intervention window?

---